
# 14 — Fattore-Style POSet Scores  
## Diagnostic Scoring Layer for the Final 5-Variable POSet

This notebook adds a diagnostic scoring layer on top of the main profile-level POSet.

It runs after:

```text
13_Final_Visuals_2002_5Var.ipynb
```

## Important methodological note

These are **diagnostic POSet scores**, not a final Economic Resilience Index.

They should help explain the POSet structure quantitatively, but they do **not** replace:

- Hasse diagrams;
- Pareto/frontier analysis;
- POSet layers;
- incomparability interpretation;
- recovery validation.

## Final decisions reflected here

- Final ordering variables = **5**.
- Main profile levels = **5**.
- Main POSet = profile-level POSet from Notebook 08.
- WGI/governance is not an ordering variable.
- Recovery is not an ordering variable.
- No scalar resilience index is created.

## Diagnostic scores computed

For each observed profile:

1. **Observed dominance score**  
   Share of observed profiles dominated by this profile.

2. **Observed dominated-by score**  
   Share of observed profiles that dominate this profile.

3. **Observed incomparability score**  
   Share of observed profiles incomparable with this profile.

4. **Embedded/full-space dominance score**  
   Dominance in the full possible ordinal profile space, not only the observed countries.

5. **Embedded/full-space incomparability score**  
   Incomparability in the full possible ordinal profile space.

6. **Separation-style score**  
   A distance-like diagnostic based on the ordinal profile-space intervals between comparable profiles.

Higher dominance/separation scores indicate a structurally stronger POSet position under the selected variables.  
Higher incomparability means a forced one-dimensional ranking would be less reliable.

## Main outputs

- `fattore_profile_scores.csv`
- `fattore_country_scores.csv`
- `fattore_score_summary_by_shock.csv`
- `fattore_scoring_method_note.csv`
- `fattore_score_interpretation_table.csv`
- `germany_fattore_position_check.csv`, if Germany is in the sample
- diagnostic figures
- `14_Fattore_Style_POSet_Scores_Audit.xlsx`


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

PROFILE_POSET_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Fattore_Style_POSet_Scores"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "14_Fattore_Style_POSet_Scores"
FIGURES_DIR = PROJECT_ROOT / "Outputs" / "Figures" / "Fattore_Style_POSet_Scores"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Fattore_Style_POSet_Scores"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

FIG_DPI = 220

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Input folder:", PROFILE_POSET_DIR.resolve())
print("Processed output:", PROCESSED_DIR.resolve())
print("Figures output:", FIGURES_DIR.resolve())


Run ID: 20260624_214846
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Input folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Profile_POSet_Main
Processed output: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Fattore_Style_POSet_Scores
Figures output: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Outputs\Figures\Fattore_Style_POSet_Scores


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

MAIN_VARIABLE_SET = "baseline_5_variables"
MAIN_PROFILE_LEVELS = 5
SHOCK_IDS = ["2007", "2019"]

LEVEL_COLUMNS = [
    "debt_capacity_level_5",
    "employment_strength_level_5",
    "rd_intensity_level_5",
    "human_capital_tertiary_level_5",
    "energy_security_level_5",
]

FINAL_5_SCORE_COLUMNS = [
    "debt_capacity_score_0_100",
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
]

SCORE_FRAMING_NOTE = (
    "Fattore-style POSet scores are included as a diagnostic scoring layer. "
    "They are not used as a final scalar Economic Resilience Index."
)

INPUT_FILES = {
    "profile_nodes_with_layers": PROFILE_POSET_DIR / "profile_nodes_with_layers.csv",
    "profile_country_map_with_layers": PROFILE_POSET_DIR / "profile_country_map_with_layers.csv",
    "profile_hasse_edges": PROFILE_POSET_DIR / "profile_hasse_edges.csv",
    "profile_run_summary": PROFILE_POSET_DIR / "profile_run_summary.csv",
}

missing_inputs = [name for name, path in INPUT_FILES.items() if not path.exists()]

if missing_inputs:
    raise FileNotFoundError(
        "Missing required profile POSet input files:\n"
        + "\n".join([f"- {name}: {INPUT_FILES[name]}" for name in missing_inputs])
    )

print("Main variable set:", MAIN_VARIABLE_SET)
print("Main profile levels:", MAIN_PROFILE_LEVELS)
print("Shocks:", SHOCK_IDS)


Main variable set: baseline_5_variables
Main profile levels: 5
Shocks: ['2007', '2019']


In [3]:

# ------------------------------------------------------
# HELPERS
# ------------------------------------------------------

figure_inventory = []
table_inventory = []
matrix_inventory = []

def clean_keys(df):
    out = df.copy()

    if "shock_id" in out.columns:
        out["shock_id"] = out["shock_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

    if "Country" in out.columns:
        out["Country"] = out["Country"].astype(str).str.strip().str.upper()

    if "baseline_year" in out.columns:
        out["baseline_year"] = pd.to_numeric(out["baseline_year"], errors="coerce").astype("Int64")

    if "levels" in out.columns:
        out["levels"] = pd.to_numeric(out["levels"], errors="coerce").astype("Int64")

    return out


def save_table(df, filename, title, description):
    processed_path = PROCESSED_DIR / filename
    diagnostics_path = DIAGNOSTICS_DIR / filename
    table_path = TABLES_DIR / filename

    df.to_csv(processed_path, index=False)
    df.to_csv(diagnostics_path, index=False)
    df.to_csv(table_path, index=False)

    table_inventory.append({
        "table_file": filename,
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "title": title,
        "description": description,
        "rows": len(df),
        "columns": len(df.columns),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved table:", filename)


def save_figure(fig, filename, title, description, source_tables):
    path = FIGURES_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)

    figure_inventory.append({
        "figure_file": filename,
        "figure_path": str(path),
        "title": title,
        "description": description,
        "source_tables": source_tables,
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved figure:", filename)


def parse_profile_code(profile_code):
    text = str(profile_code).strip()

    if "-" in text:
        parts = [p for p in text.split("-") if p != ""]
    else:
        parts = list(text)

    return np.array([int(float(p)) for p in parts], dtype=int)


def build_dominance_matrix(values):
    values = np.asarray(values, dtype=int)
    n = values.shape[0]
    dominance = np.zeros((n, n), dtype=np.int8)

    for i in range(n):
        ge_all = np.all(values[i] >= values, axis=1)
        gt_one = np.any(values[i] > values, axis=1)
        dominance[i, :] = (ge_all & gt_one).astype(np.int8)

    np.fill_diagonal(dominance, 0)
    return dominance


def full_space_dominance_count(profile_values):
    levels = np.asarray(profile_values, dtype=int)
    return int(np.prod(levels) - 1)


def full_space_dominated_by_count(profile_values, max_level):
    levels = np.asarray(profile_values, dtype=int)
    return int(np.prod(max_level - levels + 1) - 1)


def full_space_incomparability_count(profile_values, max_level):
    levels = np.asarray(profile_values, dtype=int)
    variable_count = len(levels)
    full_n = int(max_level ** variable_count)

    downset_including_self = int(np.prod(levels))
    upset_including_self = int(np.prod(max_level - levels + 1))

    comparable_including_self = downset_including_self + upset_including_self - 1
    incomparable = full_n - comparable_including_self

    return int(incomparable)


def between_count_if_dominates(upper_values, lower_values):
    upper = np.asarray(upper_values, dtype=int)
    lower = np.asarray(lower_values, dtype=int)

    if not np.all(upper >= lower):
        return 0

    return int(np.prod(upper - lower + 1))


def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None



## Load profile-level POSet outputs

This notebook uses the outputs from Notebook 08:

- `profile_nodes_with_layers.csv`
- `profile_country_map_with_layers.csv`
- `profile_hasse_edges.csv`
- `profile_run_summary.csv`

It expects the final profile POSet configuration:

```text
baseline_5_variables
5 profile levels
2007 and 2019 shock-specific cross-sections
```


In [4]:

# ------------------------------------------------------
# LOAD INPUTS
# ------------------------------------------------------

profile_nodes = clean_keys(pd.read_csv(INPUT_FILES["profile_nodes_with_layers"]))
profile_country_map = clean_keys(pd.read_csv(INPUT_FILES["profile_country_map_with_layers"]))
profile_hasse_edges = clean_keys(pd.read_csv(INPUT_FILES["profile_hasse_edges"]))
profile_run_summary = clean_keys(pd.read_csv(INPUT_FILES["profile_run_summary"]))

required_node_cols = {"shock_id", "baseline_year", "profile_code", "countries", "country_count", "layer", "is_pareto_frontier"}
missing_node_cols = required_node_cols - set(profile_nodes.columns)

if missing_node_cols:
    raise ValueError(f"profile_nodes_with_layers is missing required columns: {missing_node_cols}")

for col in LEVEL_COLUMNS:
    if col not in profile_nodes.columns:
        raise ValueError(f"Missing profile level column in profile_nodes: {col}")
    profile_nodes[col] = pd.to_numeric(profile_nodes[col], errors="coerce").astype(int)

for col in LEVEL_COLUMNS:
    if col in profile_country_map.columns:
        profile_country_map[col] = pd.to_numeric(profile_country_map[col], errors="coerce").astype(int)

selected_profile_nodes = profile_nodes[
    profile_nodes["shock_id"].isin(SHOCK_IDS)
].copy()

if "main_variable_set" in selected_profile_nodes.columns:
    selected_profile_nodes = selected_profile_nodes[
        selected_profile_nodes["main_variable_set"].eq(MAIN_VARIABLE_SET)
    ].copy()

if "levels" in selected_profile_nodes.columns:
    selected_profile_nodes = selected_profile_nodes[
        selected_profile_nodes["levels"].eq(MAIN_PROFILE_LEVELS)
    ].copy()

selected_profile_country_map = profile_country_map[
    profile_country_map["shock_id"].isin(SHOCK_IDS)
].copy()

if "main_variable_set" in selected_profile_country_map.columns:
    selected_profile_country_map = selected_profile_country_map[
        selected_profile_country_map["main_variable_set"].eq(MAIN_VARIABLE_SET)
    ].copy()

if "levels" in selected_profile_country_map.columns:
    selected_profile_country_map = selected_profile_country_map[
        selected_profile_country_map["levels"].eq(MAIN_PROFILE_LEVELS)
    ].copy()

if selected_profile_nodes.empty:
    raise ValueError("No selected profile nodes found. Check MAIN_VARIABLE_SET / MAIN_PROFILE_LEVELS.")

print("Selected profile nodes:", len(selected_profile_nodes))
print("Selected country-profile rows:", len(selected_profile_country_map))
display(selected_profile_nodes.head())
display(selected_profile_country_map.head())


Selected profile nodes: 59
Selected country-profile rows: 60


,shock_id,baseline_year,profile_code,debt_capacity_level_5,employment_strength_level_5,rd_intensity_level_5,human_capital_tertiary_level_5,energy_security_level_5,countries,country_count,profile_digit_sum,mean_debt_capacity_score,mean_employment_strength_score,mean_rd_intensity_score,mean_human_capital_tertiary_score,mean_energy_security_score,node_id,main_variable_set,levels,layer,is_pareto_frontier
0,2007,2007,4-5-5-4-5,4,5,5,4,5,DNK,1,23.0000,74.5780,100.0000,71.3466,50.0682,82.7351,2007__4-5-5-4-5,baseline_5_variables,5,1,True
1,2007,2007,5-5-2-5-5,5,5,2,5,5,EST,1,22.0000,100.0000,84.2167,21.1546,56.5490,50.2228,2007__5-5-2-5-5,baseline_5_variables,5,1,True
2,2007,2007,2-4-5-5-4,2,4,5,5,4,USA,1,20.0000,48.4466,83.6932,75.0128,77.1390,47.1942,2007__2-4-5-5-4,baseline_5_variables,5,1,True
3,2007,2007,3-3-4-5-5,3,3,4,5,5,CAN,1,20.0000,64.9865,63.6697,50.3901,100.0000,100.0000,2007__3-3-4-5-5,baseline_5_variables,5,1,True
4,2007,2007,3-5-4-4-4,3,5,4,4,4,NLD,1,20.0000,61.3704,89.8050,41.9835,49.7807,36.9664,2007__3-5-4-4-4,baseline_5_variables,5,2,False


,Country,shock_id,baseline_year,profile_code,profile_digit_sum,debt_capacity_level_5,employment_strength_level_5,rd_intensity_level_5,human_capital_tertiary_level_5,energy_security_level_5,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,main_variable_set,levels,layer,is_pareto_frontier
0,AUT,2007,2007,2-4-5-3-2,16.0000,2,4,5,3,2,38.5303,80.6046,68.7303,33.2757,22.6201,baseline_5_variables,5,2,False
1,BEL,2007,2007,1-2-4-4-1,12.0000,1,2,4,4,1,16.9811,46.5777,48.5360,53.5113,9.6139,baseline_5_variables,5,3,False
2,CAN,2007,2007,3-3-4-5-5,20.0000,3,3,4,5,5,64.9865,63.6697,50.3901,100.0000,100.0000,baseline_5_variables,5,1,True
3,CZE,2007,2007,4-3-3-1-5,16.0000,4,3,3,1,5,76.7627,74.6107,29.3904,0.4466,50.5562,baseline_5_variables,5,2,False
4,DEU,2007,2007,2-1-4-2-3,12.0000,2,1,4,2,3,40.6157,31.1478,68.2315,30.9850,28.5833,baseline_5_variables,5,4,False


In [5]:

# ------------------------------------------------------
# FATTORE-STYLE SCORING FUNCTION
# ------------------------------------------------------

def compute_fattore_style_scores_for_group(group, max_level):
    group = group.copy().reset_index(drop=True)

    profile_values = np.vstack(group["profile_code"].apply(parse_profile_code).values)
    profile_codes = group["profile_code"].astype(str).tolist()

    n_obs = len(group)
    variable_count = profile_values.shape[1]
    full_space_n = int(max_level ** variable_count)

    dominance = build_dominance_matrix(profile_values)

    dominates = dominance.sum(axis=1)
    dominated_by = dominance.sum(axis=0)

    comparable = ((dominance + dominance.T) > 0).astype(np.int8)
    incomparable = np.ones((n_obs, n_obs), dtype=np.int8) - comparable
    np.fill_diagonal(incomparable, 0)

    incomparable_count = incomparable.sum(axis=1)

    separation_observed = np.zeros((n_obs, n_obs), dtype=float)

    for i in range(n_obs):
        for j in range(n_obs):
            if i == j:
                continue
            separation_observed[i, j] = between_count_if_dominates(
                profile_values[i],
                profile_values[j],
            )

    observed_separation_raw = separation_observed.sum(axis=1)

    ideal_top = np.repeat(max_level, variable_count)
    ideal_separation_denominator = sum(
        between_count_if_dominates(ideal_top, profile_values[j])
        for j in range(n_obs)
    )

    embedded_dominance_raw = np.array([
        full_space_dominance_count(v) for v in profile_values
    ], dtype=float)

    embedded_dominated_by_raw = np.array([
        full_space_dominated_by_count(v, max_level=max_level) for v in profile_values
    ], dtype=float)

    embedded_incomparability_raw = np.array([
        full_space_incomparability_count(v, max_level=max_level) for v in profile_values
    ], dtype=float)

    obs_den = n_obs - 1 if n_obs > 1 else 1
    full_den = full_space_n - 1 if full_space_n > 1 else 1

    scores = group.copy()

    scores["variable_count"] = variable_count
    scores["profile_levels"] = max_level
    scores["full_profile_space_size"] = full_space_n

    scores["observed_dominates_count"] = dominates.astype(int)
    scores["observed_dominated_by_count"] = dominated_by.astype(int)
    scores["observed_incomparable_with_count"] = incomparable_count.astype(int)

    scores["observed_dominance_score_normalized"] = dominates / obs_den
    scores["observed_dominated_by_score_normalized"] = dominated_by / obs_den
    scores["observed_incomparability_score_normalized"] = incomparable_count / obs_den

    scores["embedded_dominance_raw"] = embedded_dominance_raw
    scores["embedded_dominated_by_raw"] = embedded_dominated_by_raw
    scores["embedded_incomparability_raw"] = embedded_incomparability_raw

    scores["embedded_dominance_score_normalized"] = embedded_dominance_raw / full_den
    scores["embedded_dominated_by_score_normalized"] = embedded_dominated_by_raw / full_den
    scores["embedded_incomparability_score_normalized"] = embedded_incomparability_raw / full_den

    scores["observed_separation_raw"] = observed_separation_raw

    if observed_separation_raw.max() > observed_separation_raw.min():
        scores["observed_separation_score_normalized"] = (
            (observed_separation_raw - observed_separation_raw.min())
            / (observed_separation_raw.max() - observed_separation_raw.min())
        )
    else:
        scores["observed_separation_score_normalized"] = 0.0

    if ideal_separation_denominator > 0:
        scores["embedded_separation_score_normalized"] = observed_separation_raw / ideal_separation_denominator
    else:
        scores["embedded_separation_score_normalized"] = np.nan

    scores["score_role"] = "diagnostic_poset_score_not_index"

    dominance_df = pd.DataFrame(dominance, index=profile_codes, columns=profile_codes)
    incomparability_df = pd.DataFrame(incomparable, index=profile_codes, columns=profile_codes)
    separation_df = pd.DataFrame(separation_observed, index=profile_codes, columns=profile_codes)

    matrices = {
        "observed_dominance_matrix": dominance_df,
        "observed_incomparability_matrix": incomparability_df,
        "observed_separation_matrix": separation_df,
    }

    return scores, matrices


In [6]:

# ------------------------------------------------------
# RUN SCORES FOR EACH SHOCK
# ------------------------------------------------------

profile_score_tables = []

for shock_id in SHOCK_IDS:
    group = selected_profile_nodes[
        selected_profile_nodes["shock_id"].astype(str).eq(str(shock_id))
    ].copy()

    if group.empty:
        print(f"No profiles for shock {shock_id}. Skipping.")
        continue

    scores, matrices = compute_fattore_style_scores_for_group(
        group=group,
        max_level=MAIN_PROFILE_LEVELS,
    )

    profile_score_tables.append(scores)

    for matrix_name, matrix_df in matrices.items():
        matrix_file = f"{matrix_name}_shock_{shock_id}_{MAIN_PROFILE_LEVELS}levels.csv"
        matrix_df.to_csv(PROCESSED_DIR / matrix_file)
        matrix_df.to_csv(DIAGNOSTICS_DIR / matrix_file)
        matrix_df.to_csv(TABLES_DIR / matrix_file)

        matrix_inventory.append({
            "shock_id": shock_id,
            "matrix": matrix_name,
            "file": matrix_file,
            "rows": len(matrix_df),
            "columns": len(matrix_df.columns),
            "created_at": RUN_TIMESTAMP,
        })

        print("Saved matrix:", matrix_file)

fattore_profile_scores = pd.concat(profile_score_tables, ignore_index=True)

save_table(
    fattore_profile_scores,
    "fattore_profile_scores.csv",
    "Fattore-style profile scores",
    "Observed and embedded POSet dominance, incomparability, and separation-style profile scores.",
)

display(fattore_profile_scores.head())


Saved matrix: observed_dominance_matrix_shock_2007_5levels.csv
Saved matrix: observed_incomparability_matrix_shock_2007_5levels.csv
Saved matrix: observed_separation_matrix_shock_2007_5levels.csv
Saved matrix: observed_dominance_matrix_shock_2019_5levels.csv
Saved matrix: observed_incomparability_matrix_shock_2019_5levels.csv
Saved matrix: observed_separation_matrix_shock_2019_5levels.csv
Saved table: fattore_profile_scores.csv


,shock_id,baseline_year,profile_code,debt_capacity_level_5,employment_strength_level_5,rd_intensity_level_5,human_capital_tertiary_level_5,energy_security_level_5,countries,country_count,profile_digit_sum,mean_debt_capacity_score,mean_employment_strength_score,mean_rd_intensity_score,mean_human_capital_tertiary_score,mean_energy_security_score,node_id,main_variable_set,levels,layer,is_pareto_frontier,variable_count,profile_levels,full_profile_space_size,observed_dominates_count,observed_dominated_by_count,observed_incomparable_with_count,observed_dominance_score_normalized,observed_dominated_by_score_normalized,observed_incomparability_score_normalized,embedded_dominance_raw,embedded_dominated_by_raw,embedded_incomparability_raw,embedded_dominance_score_normalized,embedded_dominated_by_score_normalized,embedded_incomparability_score_normalized,observed_separation_raw,observed_separation_score_normalized,embedded_separation_score_normalized,score_role
0,2007,2007,4-5-5-4-5,4,5,5,4,5,DNK,1,23.0000,74.5780,100.0000,71.3466,50.0682,82.7351,2007__4-5-5-4-5,baseline_5_variables,5,1,True,5,5,3125,15,0,9,0.6250,0.0000,0.3750,1999.0000,3.0000,1122.0000,0.6399,0.0010,0.3592,5392.0000,1.0000,0.5250,diagnostic_poset_score_not_index
1,2007,2007,5-5-2-5-5,5,5,2,5,5,EST,1,22.0000,100.0000,84.2167,21.1546,56.5490,50.2228,2007__5-5-2-5-5,baseline_5_variables,5,1,True,5,5,3125,9,0,15,0.3750,0.0000,0.6250,1249.0000,3.0000,1872.0000,0.3998,0.0010,0.5992,2634.0000,0.4885,0.2565,diagnostic_poset_score_not_index
2,2007,2007,2-4-5-5-4,2,4,5,5,4,USA,1,20.0000,48.4466,83.6932,75.0128,77.1390,47.1942,2007__2-4-5-5-4,baseline_5_variables,5,1,True,5,5,3125,8,0,16,0.3333,0.0000,0.6667,799.0000,15.0000,2310.0000,0.2558,0.0048,0.7394,1525.0000,0.2828,0.1485,diagnostic_poset_score_not_index
3,2007,2007,3-3-4-5-5,3,3,4,5,5,CAN,1,20.0000,64.9865,63.6697,50.3901,100.0000,100.0000,2007__3-3-4-5-5,baseline_5_variables,5,1,True,5,5,3125,8,0,16,0.3333,0.0000,0.6667,899.0000,17.0000,2208.0000,0.2878,0.0054,0.7068,1650.0000,0.3060,0.1607,diagnostic_poset_score_not_index
4,2007,2007,3-5-4-4-4,3,5,4,4,4,NLD,1,20.0000,61.3704,89.8050,41.9835,49.7807,36.9664,2007__3-5-4-4-4,baseline_5_variables,5,2,False,5,5,3125,8,1,15,0.3333,0.0417,0.6250,959.0000,23.0000,2142.0000,0.3070,0.0074,0.6857,1832.0000,0.3398,0.1784,diagnostic_poset_score_not_index


In [7]:

# ------------------------------------------------------
# COUNTRY-LEVEL SCORE MAPPING
# ------------------------------------------------------

score_cols_for_merge = [
    "shock_id", "baseline_year", "profile_code",
    "layer", "is_pareto_frontier",
    "observed_dominance_score_normalized",
    "observed_dominated_by_score_normalized",
    "observed_incomparability_score_normalized",
    "embedded_dominance_score_normalized",
    "embedded_dominated_by_score_normalized",
    "embedded_incomparability_score_normalized",
    "observed_separation_score_normalized",
    "embedded_separation_score_normalized",
    "score_role",
]

score_cols_for_merge = [c for c in score_cols_for_merge if c in fattore_profile_scores.columns]

fattore_country_scores = selected_profile_country_map.merge(
    fattore_profile_scores[score_cols_for_merge],
    on=["shock_id", "baseline_year", "profile_code"],
    how="left",
    suffixes=("", "_score"),
)

# Clean duplicated columns from merge.
for base_col in ["layer", "is_pareto_frontier"]:
    score_col = f"{base_col}_score"
    if score_col in fattore_country_scores.columns:
        if base_col not in fattore_country_scores.columns:
            fattore_country_scores[base_col] = fattore_country_scores[score_col]
        fattore_country_scores.drop(columns=[score_col], inplace=True)

preferred_first_cols = [
    "shock_id", "baseline_year", "Country", "profile_code",
    "layer", "is_pareto_frontier",
    "observed_dominance_score_normalized",
    "observed_dominated_by_score_normalized",
    "observed_incomparability_score_normalized",
    "embedded_dominance_score_normalized",
    "embedded_dominated_by_score_normalized",
    "embedded_incomparability_score_normalized",
    "observed_separation_score_normalized",
    "embedded_separation_score_normalized",
    "score_role",
]

ordered_cols = [c for c in preferred_first_cols if c in fattore_country_scores.columns]
ordered_cols += [c for c in fattore_country_scores.columns if c not in ordered_cols]

fattore_country_scores = fattore_country_scores[ordered_cols].copy()

save_table(
    fattore_country_scores,
    "fattore_country_scores.csv",
    "Fattore-style country scores",
    "Country-level mapping of diagnostic Fattore-style scores from final profiles.",
)

display(fattore_country_scores.head())


Saved table: fattore_country_scores.csv


,shock_id,baseline_year,Country,profile_code,layer,is_pareto_frontier,observed_dominance_score_normalized,observed_dominated_by_score_normalized,observed_incomparability_score_normalized,embedded_dominance_score_normalized,embedded_dominated_by_score_normalized,embedded_incomparability_score_normalized,observed_separation_score_normalized,embedded_separation_score_normalized,score_role,profile_digit_sum,debt_capacity_level_5,employment_strength_level_5,rd_intensity_level_5,human_capital_tertiary_level_5,energy_security_level_5,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,main_variable_set,levels
0,2007,2007,AUT,2-4-5-3-2,2,False,0.1250,0.0833,0.7917,0.0765,0.0304,0.8931,0.0504,0.0265,diagnostic_poset_score_not_index,16.0000,2,4,5,3,2,38.5303,80.6046,68.7303,33.2757,22.6201,baseline_5_variables,5
1,2007,2007,BEL,1-2-4-4-1,3,False,0.0417,0.2500,0.7083,0.0099,0.1277,0.8624,0.0022,0.0012,diagnostic_poset_score_not_index,12.0000,1,2,4,4,1,16.9811,46.5777,48.5360,53.5113,9.6139,baseline_5_variables,5
2,2007,2007,CAN,3-3-4-5-5,1,True,0.3333,0.0000,0.6667,0.2878,0.0054,0.7068,0.3060,0.1607,diagnostic_poset_score_not_index,20.0000,3,3,4,5,5,64.9865,63.6697,50.3901,100.0000,100.0000,baseline_5_variables,5
3,2007,2007,CZE,4-3-3-1-5,2,False,0.1667,0.0417,0.7917,0.0573,0.0285,0.9142,0.0341,0.0179,diagnostic_poset_score_not_index,16.0000,4,3,3,1,5,76.7627,74.6107,29.3904,0.4466,50.5562,baseline_5_variables,5
4,2007,2007,DEU,2-1-4-2-3,4,False,0.0417,0.2917,0.6667,0.0150,0.1533,0.8316,0.0030,0.0016,diagnostic_poset_score_not_index,12.0000,2,1,4,2,3,40.6157,31.1478,68.2315,30.9850,28.5833,baseline_5_variables,5


In [8]:

# ------------------------------------------------------
# SCORE SUMMARY BY SHOCK + INTERPRETATION TABLE
# ------------------------------------------------------

summary_rows = []

for shock_id, group in fattore_profile_scores.groupby("shock_id"):
    row = {
        "shock_id": shock_id,
        "profile_count": len(group),
        "profile_levels": MAIN_PROFILE_LEVELS,
        "variable_count": len(LEVEL_COLUMNS),
        "main_variable_set": MAIN_VARIABLE_SET,
        "mean_observed_dominance_score": group["observed_dominance_score_normalized"].mean(),
        "mean_observed_incomparability_score": group["observed_incomparability_score_normalized"].mean(),
        "mean_embedded_dominance_score": group["embedded_dominance_score_normalized"].mean(),
        "mean_embedded_incomparability_score": group["embedded_incomparability_score_normalized"].mean(),
        "mean_embedded_separation_score": group["embedded_separation_score_normalized"].mean(),
        "max_embedded_separation_score": group["embedded_separation_score_normalized"].max(),
        "pareto_profile_count": int(group["is_pareto_frontier"].sum()) if "is_pareto_frontier" in group.columns else np.nan,
        "note": SCORE_FRAMING_NOTE,
    }
    summary_rows.append(row)

fattore_score_summary_by_shock = pd.DataFrame(summary_rows)

fattore_score_interpretation_table = pd.DataFrame([
    {
        "score_name": "observed_dominance_score_normalized",
        "meaning": "Share of observed profiles dominated by the profile.",
        "interpretation": "Higher values indicate a structurally stronger observed profile position.",
        "index_warning": "Diagnostic only; do not average into final index.",
    },
    {
        "score_name": "observed_incomparability_score_normalized",
        "meaning": "Share of observed profiles incomparable with the profile.",
        "interpretation": "Higher values indicate that forced ranking is less reliable.",
        "index_warning": "Incomparability is a substantive POSet result, not a problem to hide.",
    },
    {
        "score_name": "embedded_dominance_score_normalized",
        "meaning": "Profile dominance share in the full possible ordinal profile space.",
        "interpretation": "Higher values indicate stronger theoretical POSet position in the product order.",
        "index_warning": "Diagnostic only; depends on chosen ordinal levels and variables.",
    },
    {
        "score_name": "embedded_incomparability_score_normalized",
        "meaning": "Profile incomparability share in the full possible ordinal profile space.",
        "interpretation": "Higher values indicate more multidimensional trade-off structure.",
        "index_warning": "Do not interpret as bad/good without context.",
    },
    {
        "score_name": "embedded_separation_score_normalized",
        "meaning": "Distance-like diagnostic based on profile-space intervals to dominated profiles.",
        "interpretation": "Higher values indicate stronger separation from structurally lower profiles.",
        "index_warning": "Useful for explanation, not for creating a final ranking.",
    },
])

method_note = pd.DataFrame([
    {
        "topic": "Scope",
        "note": "This notebook adds Fattore-style diagnostic scores on top of the profile POSet. It does not replace the Hasse diagrams or create a final Economic Resilience Index.",
    },
    {
        "topic": "Observed dominance",
        "note": "Observed dominance score is the share of observed profiles dominated by a profile.",
    },
    {
        "topic": "Observed incomparability",
        "note": "Observed incomparability score is the share of observed profiles incomparable with a profile.",
    },
    {
        "topic": "Embedded/full-space scoring",
        "note": "Embedded scores use the full ordinal profile space implied by the five variables and five profile levels, not only the observed countries.",
    },
    {
        "topic": "Separation-style score",
        "note": "The separation-style score counts ordinal profile-space intervals between comparable profiles. It is a distance-like diagnostic score.",
    },
    {
        "topic": "Interpretation warning",
        "note": "Higher dominance or separation scores help explain POSet position. They should not be presented as a final scalar resilience index.",
    },
])

save_table(
    fattore_score_summary_by_shock,
    "fattore_score_summary_by_shock.csv",
    "Fattore-style score summary by shock",
    "Shock-level summary of diagnostic dominance, incomparability, and separation-style scores.",
)

save_table(
    fattore_score_interpretation_table,
    "fattore_score_interpretation_table.csv",
    "Fattore-style score interpretation table",
    "Interpretation guide for diagnostic POSet scores.",
)

save_table(
    method_note,
    "fattore_scoring_method_note.csv",
    "Fattore-style scoring method note",
    "Short methodological note explaining diagnostic score definitions.",
)

display(fattore_score_summary_by_shock)
display(fattore_score_interpretation_table)
display(method_note)


Saved table: fattore_score_summary_by_shock.csv
Saved table: fattore_score_interpretation_table.csv
Saved table: fattore_scoring_method_note.csv


,shock_id,profile_count,profile_levels,variable_count,main_variable_set,mean_observed_dominance_score,mean_observed_incomparability_score,mean_embedded_dominance_score,mean_embedded_incomparability_score,mean_embedded_separation_score,max_embedded_separation_score,pareto_profile_count,note
0,2007,25,5,5,baseline_5_variables,0.1600,0.6800,0.1182,0.7507,0.0666,0.5250,8,Fattore-style POSet scores are included as a d...
1,2019,34,5,5,baseline_5_variables,0.1150,0.7701,0.0903,0.7969,0.0398,0.2489,12,Fattore-style POSet scores are included as a d...


,score_name,meaning,interpretation,index_warning
0,observed_dominance_score_normalized,Share of observed profiles dominated by the pr...,Higher values indicate a structurally stronger...,Diagnostic only; do not average into final index.
1,observed_incomparability_score_normalized,Share of observed profiles incomparable with t...,Higher values indicate that forced ranking is ...,"Incomparability is a substantive POSet result,..."
2,embedded_dominance_score_normalized,Profile dominance share in the full possible o...,Higher values indicate stronger theoretical PO...,Diagnostic only; depends on chosen ordinal lev...
3,embedded_incomparability_score_normalized,Profile incomparability share in the full poss...,Higher values indicate more multidimensional t...,Do not interpret as bad/good without context.
4,embedded_separation_score_normalized,Distance-like diagnostic based on profile-spac...,Higher values indicate stronger separation fro...,"Useful for explanation, not for creating a fin..."


,topic,note
0,Scope,This notebook adds Fattore-style diagnostic sc...
1,Observed dominance,Observed dominance score is the share of obser...
2,Observed incomparability,Observed incomparability score is the share of...
3,Embedded/full-space scoring,Embedded scores use the full ordinal profile s...
4,Separation-style score,The separation-style score counts ordinal prof...
5,Interpretation warning,Higher dominance or separation scores help exp...


In [9]:

# ------------------------------------------------------
# REPORT-READY FATTORE SCORE PARAGRAPHS
# ------------------------------------------------------

report_ready_fattore_score_paragraphs = pd.DataFrame([
    {
        "topic": "Purpose of diagnostic scores",
        "report_text": (
            "A Fattore-style diagnostic scoring layer was added after the main POSet construction to summarize "
            "dominance, incomparability, and profile-space separation. These scores are used only to interpret "
            "the partial-order structure and are not combined into an Economic Resilience Index."
        ),
    },
    {
        "topic": "Observed versus embedded scores",
        "report_text": (
            "Observed scores describe relations among the profiles that actually appear in the sample. Embedded "
            "scores instead compare a profile with the full theoretical ordinal profile space implied by five "
            "variables and five levels."
        ),
    },
    {
        "topic": "Incomparability interpretation",
        "report_text": (
            "The incomparability score is important because it quantifies how often a profile cannot be ordered "
            "against other profiles without making arbitrary trade-offs. High incomparability supports the use "
            "of POSet rather than a forced scalar ranking."
        ),
    },
    {
        "topic": "No scalar index",
        "report_text": (
            "Although the diagnostic scores are numeric, they are not used to create a single final ranking. "
            "The main result remains the Hasse diagram, layer structure, Pareto/frontier profiles, and recovery "
            "validation."
        ),
    },
])

save_table(
    report_ready_fattore_score_paragraphs,
    "report_ready_fattore_score_paragraphs.csv",
    "Report-ready Fattore score paragraphs",
    "Report-ready explanatory text for diagnostic POSet scores.",
)

display(report_ready_fattore_score_paragraphs)


Saved table: report_ready_fattore_score_paragraphs.csv


,topic,report_text
0,Purpose of diagnostic scores,A Fattore-style diagnostic scoring layer was a...
1,Observed versus embedded scores,Observed scores describe relations among the p...
2,Incomparability interpretation,The incomparability score is important because...
3,No scalar index,"Although the diagnostic scores are numeric, th..."


In [10]:

# ------------------------------------------------------
# FIGURE 01 — OBSERVED DOMINANCE VS INCOMPARABILITY
# ------------------------------------------------------

for shock_id, group in fattore_profile_scores.groupby("shock_id"):
    fig, ax = plt.subplots(figsize=(9, 6.5))

    ax.scatter(
        group["observed_incomparability_score_normalized"],
        group["observed_dominance_score_normalized"],
        s=90,
        alpha=0.8,
    )

    for _, row in group.iterrows():
        label = f"{row['profile_code']}\n{str(row.get('countries', ''))[:24]}"
        ax.text(
            row["observed_incomparability_score_normalized"],
            row["observed_dominance_score_normalized"],
            label,
            fontsize=7,
            ha="center",
            va="bottom",
        )

    ax.set_title(f"Observed dominance vs incomparability — shock {shock_id}")
    ax.set_xlabel("Observed incomparability score")
    ax.set_ylabel("Observed dominance score")
    ax.grid(alpha=0.25)

    save_figure(
        fig,
        f"01_observed_dominance_vs_incomparability_shock_{shock_id}_5var_5levels.png",
        f"Observed dominance vs incomparability — {shock_id}",
        "Scatter plot of observed profile dominance and incomparability scores.",
        "fattore_profile_scores.csv",
    )


Saved figure: 01_observed_dominance_vs_incomparability_shock_2007_5var_5levels.png
Saved figure: 01_observed_dominance_vs_incomparability_shock_2019_5var_5levels.png


In [11]:

# ------------------------------------------------------
# FIGURE 02 — EMBEDDED DOMINANCE VS SEPARATION
# ------------------------------------------------------

for shock_id, group in fattore_profile_scores.groupby("shock_id"):
    fig, ax = plt.subplots(figsize=(9, 6.5))

    ax.scatter(
        group["embedded_dominance_score_normalized"],
        group["embedded_separation_score_normalized"],
        s=90,
        alpha=0.8,
    )

    for _, row in group.iterrows():
        label = f"{row['profile_code']}\n{str(row.get('countries', ''))[:24]}"
        ax.text(
            row["embedded_dominance_score_normalized"],
            row["embedded_separation_score_normalized"],
            label,
            fontsize=7,
            ha="center",
            va="bottom",
        )

    ax.set_title(f"Embedded dominance vs separation-style score — shock {shock_id}")
    ax.set_xlabel("Embedded dominance score")
    ax.set_ylabel("Embedded separation-style score")
    ax.grid(alpha=0.25)

    save_figure(
        fig,
        f"02_embedded_dominance_vs_separation_shock_{shock_id}_5var_5levels.png",
        f"Embedded dominance vs separation — {shock_id}",
        "Scatter plot comparing full-space embedded dominance and separation-style scores.",
        "fattore_profile_scores.csv",
    )


Saved figure: 02_embedded_dominance_vs_separation_shock_2007_5var_5levels.png
Saved figure: 02_embedded_dominance_vs_separation_shock_2019_5var_5levels.png


In [12]:

# ------------------------------------------------------
# FIGURE 03 — COUNTRY DIAGNOSTIC SEPARATION SCORE DISPLAY
# ------------------------------------------------------

for shock_id, group in fattore_country_scores.groupby("shock_id"):
    plot_df = group.copy()

    if "embedded_separation_score_normalized" not in plot_df.columns:
        continue

    plot_df = plot_df.sort_values(
        ["embedded_separation_score_normalized", "observed_incomparability_score_normalized", "Country"],
        ascending=[False, True, True],
    )

    fig_height = max(6, len(plot_df) * 0.28)
    fig, ax = plt.subplots(figsize=(11, fig_height))

    labels = plot_df["Country"].astype(str).tolist()
    y = np.arange(len(plot_df))

    ax.barh(y, plot_df["embedded_separation_score_normalized"])
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("Embedded separation-style score")
    ax.set_title(
        f"Country diagnostic separation-style scores — shock {shock_id}\n"
        "Diagnostic POSet score, not a final Economic Resilience Index"
    )
    ax.grid(axis="x", alpha=0.25)

    save_figure(
        fig,
        f"03_country_diagnostic_separation_scores_shock_{shock_id}_5var_5levels.png",
        f"Country diagnostic separation-style scores — {shock_id}",
        "Country-level diagnostic display of embedded separation-style profile scores. Not a final ranking/index.",
        "fattore_country_scores.csv",
    )


Saved figure: 03_country_diagnostic_separation_scores_shock_2007_5var_5levels.png
Saved figure: 03_country_diagnostic_separation_scores_shock_2019_5var_5levels.png


In [13]:

# ------------------------------------------------------
# SPECIAL CHECK — GERMANY / DEU POSITION
# ------------------------------------------------------
# This check explains that POSet position is shock-specific and profile-specific.

germany_check = pd.DataFrame()

if "Country" in fattore_country_scores.columns:
    germany_check = fattore_country_scores[
        fattore_country_scores["Country"].astype(str).str.upper().isin(["DEU", "GER", "GERMANY"])
    ].copy()

    if not germany_check.empty:
        selected_cols = [
            "shock_id", "baseline_year", "Country", "profile_code",
            "layer", "is_pareto_frontier",
            "observed_dominance_score_normalized",
            "observed_dominated_by_score_normalized",
            "observed_incomparability_score_normalized",
            "embedded_dominance_score_normalized",
            "embedded_dominated_by_score_normalized",
            "embedded_incomparability_score_normalized",
            "embedded_separation_score_normalized",
        ]

        selected_cols = [c for c in selected_cols if c in germany_check.columns]
        germany_check = germany_check[selected_cols].copy()

        save_table(
            germany_check,
            "germany_fattore_position_check.csv",
            "Germany Fattore-style position check",
            "Diagnostic check showing Germany's profile position by shock.",
        )

        display(germany_check)
    else:
        print("Germany / DEU not found in selected country scores.")
else:
    print("No Country column found in fattore_country_scores.")


Saved table: germany_fattore_position_check.csv


,shock_id,baseline_year,Country,profile_code,layer,is_pareto_frontier,observed_dominance_score_normalized,observed_dominated_by_score_normalized,observed_incomparability_score_normalized,embedded_dominance_score_normalized,embedded_dominated_by_score_normalized,embedded_incomparability_score_normalized,embedded_separation_score_normalized
4,2007,2007,DEU,2-1-4-2-3,4,False,0.0417,0.2917,0.6667,0.0150,0.1533,0.8316,0.0016
33,2019,2019,DEU,3-5-5-2-2,1,True,0.1212,0.0000,0.8788,0.0957,0.0150,0.8892,0.0315


In [14]:

# ------------------------------------------------------
# INVENTORIES AND AUDIT WORKBOOK
# ------------------------------------------------------

figure_inventory_df = pd.DataFrame(figure_inventory)
table_inventory_df = pd.DataFrame(table_inventory)
matrix_inventory_df = pd.DataFrame(matrix_inventory)

for table, file_name in [
    (figure_inventory_df, "fattore_figure_inventory.csv"),
    (table_inventory_df, "fattore_table_inventory.csv"),
    (matrix_inventory_df, "fattore_matrix_inventory.csv"),
]:
    table.to_csv(PROCESSED_DIR / file_name, index=False)
    table.to_csv(DIAGNOSTICS_DIR / file_name, index=False)
    table.to_csv(TABLES_DIR / file_name, index=False)

run_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "run_timestamp": RUN_TIMESTAMP,
    "main_variable_set": MAIN_VARIABLE_SET,
    "profile_levels": MAIN_PROFILE_LEVELS,
    "shock_ids": ",".join(SHOCK_IDS),
    "profile_score_rows": len(fattore_profile_scores),
    "country_score_rows": len(fattore_country_scores),
    "figure_count": len(figure_inventory_df),
    "table_count": len(table_inventory_df),
    "matrix_count": len(matrix_inventory_df),
    "processed_dir": str(PROCESSED_DIR),
    "figures_dir": str(FIGURES_DIR),
    "tables_dir": str(TABLES_DIR),
    "note": SCORE_FRAMING_NOTE,
}])

run_summary.to_csv(PROCESSED_DIR / "fattore_run_summary.csv", index=False)
run_summary.to_csv(DIAGNOSTICS_DIR / "fattore_run_summary.csv", index=False)
run_summary.to_csv(TABLES_DIR / "fattore_run_summary.csv", index=False)

audit_xlsx = AUDIT_DIR / "14_Fattore_Style_POSet_Scores_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    run_summary.to_excel(writer, sheet_name="run_summary", index=False)
    fattore_score_summary_by_shock.to_excel(writer, sheet_name="score_summary", index=False)
    fattore_score_interpretation_table.to_excel(writer, sheet_name="score_interpretation", index=False)
    method_note.to_excel(writer, sheet_name="method_note", index=False)
    table_inventory_df.to_excel(writer, sheet_name="table_inventory", index=False)
    figure_inventory_df.to_excel(writer, sheet_name="figure_inventory", index=False)
    matrix_inventory_df.to_excel(writer, sheet_name="matrix_inventory", index=False)
    report_ready_fattore_score_paragraphs.to_excel(writer, sheet_name="report_paragraphs", index=False)

    if isinstance(germany_check, pd.DataFrame) and not germany_check.empty:
        germany_check.to_excel(writer, sheet_name="germany_check", index=False)

print("Audit workbook:", audit_xlsx.resolve())
display(run_summary)
display(table_inventory_df)
display(figure_inventory_df)


Audit workbook: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\14_Fattore_Style_POSet_Scores_Audit.xlsx


,run_id,run_timestamp,main_variable_set,profile_levels,shock_ids,profile_score_rows,country_score_rows,figure_count,table_count,matrix_count,processed_dir,figures_dir,tables_dir,note
0,20260624_214846,2026-06-24 21:48:46,baseline_5_variables,5,"2007,2019",59,60,6,7,6,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,Fattore-style POSet scores are included as a d...


,table_file,processed_path,diagnostics_path,table_path,title,description,rows,columns,created_at
0,fattore_profile_scores.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,Fattore-style profile scores,"Observed and embedded POSet dominance, incompa...",59,40,2026-06-24 21:48:46
1,fattore_country_scores.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,Fattore-style country scores,Country-level mapping of diagnostic Fattore-st...,60,28,2026-06-24 21:48:46
2,fattore_score_summary_by_shock.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,Fattore-style score summary by shock,"Shock-level summary of diagnostic dominance, i...",2,13,2026-06-24 21:48:46
3,fattore_score_interpretation_table.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,Fattore-style score interpretation table,Interpretation guide for diagnostic POSet scores.,5,4,2026-06-24 21:48:46
4,fattore_scoring_method_note.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,Fattore-style scoring method note,Short methodological note explaining diagnosti...,6,2,2026-06-24 21:48:46
5,report_ready_fattore_score_paragraphs.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,Report-ready Fattore score paragraphs,Report-ready explanatory text for diagnostic P...,4,2,2026-06-24 21:48:46
6,germany_fattore_position_check.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,Germany Fattore-style position check,Diagnostic check showing Germany's profile pos...,2,13,2026-06-24 21:48:46


,figure_file,figure_path,title,description,source_tables,created_at
0,01_observed_dominance_vs_incomparability_shock...,D:\Milano Bicocca\Course Materials\1st Year\02...,Observed dominance vs incomparability — 2007,Scatter plot of observed profile dominance and...,fattore_profile_scores.csv,2026-06-24 21:48:46
1,01_observed_dominance_vs_incomparability_shock...,D:\Milano Bicocca\Course Materials\1st Year\02...,Observed dominance vs incomparability — 2019,Scatter plot of observed profile dominance and...,fattore_profile_scores.csv,2026-06-24 21:48:46
2,02_embedded_dominance_vs_separation_shock_2007...,D:\Milano Bicocca\Course Materials\1st Year\02...,Embedded dominance vs separation — 2007,Scatter plot comparing full-space embedded dom...,fattore_profile_scores.csv,2026-06-24 21:48:46
3,02_embedded_dominance_vs_separation_shock_2019...,D:\Milano Bicocca\Course Materials\1st Year\02...,Embedded dominance vs separation — 2019,Scatter plot comparing full-space embedded dom...,fattore_profile_scores.csv,2026-06-24 21:48:46
4,03_country_diagnostic_separation_scores_shock_...,D:\Milano Bicocca\Course Materials\1st Year\02...,Country diagnostic separation-style scores — 2007,Country-level diagnostic display of embedded s...,fattore_country_scores.csv,2026-06-24 21:48:46
5,03_country_diagnostic_separation_scores_shock_...,D:\Milano Bicocca\Course Materials\1st Year\02...,Country diagnostic separation-style scores — 2019,Country-level diagnostic display of embedded s...,fattore_country_scores.csv,2026-06-24 21:48:46


In [15]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

expected_outputs = [
    "fattore_profile_scores.csv",
    "fattore_country_scores.csv",
    "fattore_score_summary_by_shock.csv",
    "fattore_score_interpretation_table.csv",
    "fattore_scoring_method_note.csv",
    "report_ready_fattore_score_paragraphs.csv",
    "fattore_run_summary.csv",
    "fattore_figure_inventory.csv",
    "fattore_table_inventory.csv",
    "fattore_matrix_inventory.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("14 FATTORE-STYLE POSET SCORES COMPLETE")
print("=" * 90)

display(output_check)

print("\nImportant interpretation:")
print("1. These are diagnostic POSet scores, not a final Economic Resilience Index.")
print("2. Dominance and separation scores help interpret the POSet quantitatively.")
print("3. Incomparability score tells us whether a forced ranking would be unreliable.")
print("4. Country positions can differ between 2007 and 2019 because profiles are shock-specific.")

print("\nKey outputs to inspect/send:")
print("- 14_Fattore_Style_POSet_Scores_Audit.xlsx")
print("- fattore_profile_scores.csv")
print("- fattore_country_scores.csv")
print("- fattore_score_summary_by_shock.csv")

print("\nNext notebook:")
print("15_Country_POSet_Diagnostic_Drilldown_2002_5Var.ipynb")


14 FATTORE-STYLE POSET SCORES COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,fattore_profile_scores.csv,True,True,True
1,fattore_country_scores.csv,True,True,True
2,fattore_score_summary_by_shock.csv,True,True,True
3,fattore_score_interpretation_table.csv,True,True,True
4,fattore_scoring_method_note.csv,True,True,True
5,report_ready_fattore_score_paragraphs.csv,True,True,True
6,fattore_run_summary.csv,True,True,True
7,fattore_figure_inventory.csv,True,True,True
8,fattore_table_inventory.csv,True,True,True
9,fattore_matrix_inventory.csv,True,True,True



Important interpretation:
1. These are diagnostic POSet scores, not a final Economic Resilience Index.
2. Dominance and separation scores help interpret the POSet quantitatively.
3. Incomparability score tells us whether a forced ranking would be unreliable.
4. Country positions can differ between 2007 and 2019 because profiles are shock-specific.

Key outputs to inspect/send:
- 14_Fattore_Style_POSet_Scores_Audit.xlsx
- fattore_profile_scores.csv
- fattore_country_scores.csv
- fattore_score_summary_by_shock.csv

Next notebook:
15_Country_POSet_Diagnostic_Drilldown_2002_5Var.ipynb
